# Train Every Finalized Study Model

This is the single training notebook for the repository. Starting from the raw
retinal images, it validates the tracked patient-disjoint manifests, builds all
20 approved representation caches, generates the locked experiment configs,
and trains the current canonical set of 24 models:

- all 20 representations with seed 42;
- full RGB with seeds 43, 44, 45, and 46 for the matched RGB4 control.

The raw images are not distributed with the repository. Put them in your own
Google Drive folder and set the three placeholders below. The repository
provides the manifests, representation contracts, train-only normalization
statistics, and training code.

Training 24 ResNet-50 models for 80 epochs will normally require multiple Colab
sessions. Run the notebook again after a disconnect: completed models are
skipped, an interrupted model resumes from `latest_checkpoint.pt`, and caches
are reused when their fingerprints are still valid.


## 1. Mount Google Drive

Use a GPU runtime. Drive stores the generated caches and model artifacts so
they survive Colab disconnections.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. User settings

Replace every `CHANGE_ME` value before running. `DATA_ROOT` must contain an
`images/` directory whose filenames match the tracked manifests. `RUN_ROOT`
must be a separate writable Drive directory with enough free space for roughly
10 GB of caches plus checkpoints.


In [ ]:
from pathlib import Path

# Repository that will be created after this cleanup is complete.
REPO_URL = "https://github.com/CHANGE_ME/retinal-color-transfer.git"
REPO_REF = "main"

# Expected example: DATA_ROOT / "images" / "img00001.jpg"
DATA_ROOT = Path("/content/drive/MyDrive/CHANGE_ME/retinal-data")

# Persistent generated caches, configs, and models are written below this root.
RUN_ROOT = Path("/content/drive/MyDrive/CHANGE_ME/retinal-color-transfer-run")

REPO_DIR = Path("/content/retinal-color-transfer")
CACHE_ROOT = RUN_ROOT / "caches"
MODEL_ROOT = RUN_ROOT / "models"
GENERATED_CONFIG_ROOT = RUN_ROOT / "configs"

BATCH_SIZE = 32
EPOCHS = 80
NUM_WORKERS = 2
PREFETCH_FACTOR = 2
MIXED_PRECISION = "fp16"
MEMORY_FORMAT = "channels_last"

SEED42_REPRESENTATIONS = (
    "rgb",
    "rgb_r",
    "rgb_g",
    "rgb_b",
    "grayscale",
    "lab",
    "lab_l",
    "lab_a",
    "lab_b",
    "hsv",
    "hsv_h",
    "hsv_s",
    "hsv_v",
    "ycrcb",
    "ycrcb_y",
    "ycrcb_cr",
    "ycrcb_cb",
    "custom_lab_b_rgb_g_rgb_b",
    "custom_lab_b_rgb_g_hsv_s",
    "custom_lab_a_rgb_g_lab_b",
)
EXTRA_RGB_SEEDS = (43, 44, 45, 46)
MODELS = (
    tuple((representation, 42) for representation in SEED42_REPRESENTATIONS)
    + tuple(("rgb", seed) for seed in EXTRA_RGB_SEEDS)
)
MODEL_NAMES = tuple(
    f"{representation}_seed{seed}" for representation, seed in MODELS
)

assert len(SEED42_REPRESENTATIONS) == 20
assert len(MODELS) == len(set(MODELS)) == 24
print(f"Configured {len(MODELS)} models.")


## 3. Clone and install the repository

The clone lives only in Colab's temporary filesystem. `RUN_ROOT` is never
deleted. Re-running this cell refreshes the code while preserving caches,
generated configs, completed models, and resumable checkpoints in Drive.


In [ ]:
import importlib
import os
import shutil
import subprocess
import sys

placeholder_values = {
    "REPO_URL": REPO_URL,
    "DATA_ROOT": str(DATA_ROOT),
    "RUN_ROOT": str(RUN_ROOT),
}
unresolved = [
    name for name, value in placeholder_values.items()
    if "CHANGE_ME" in value
]
if unresolved:
    raise ValueError(
        "Replace the placeholder value(s) before running: "
        + ", ".join(unresolved)
    )
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")
if not (DATA_ROOT / "images").is_dir():
    raise FileNotFoundError(
        f"Expected the raw image directory at {DATA_ROOT / 'images'}"
    )
if DATA_ROOT.resolve() == RUN_ROOT.resolve():
    raise ValueError("DATA_ROOT and RUN_ROOT must be different directories.")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
GENERATED_CONFIG_ROOT.mkdir(parents=True, exist_ok=True)

total, used, free = shutil.disk_usage(RUN_ROOT)
free_gib = free / (1024 ** 3)
print(f"Free space at RUN_ROOT: {free_gib:.1f} GiB")
if free_gib < 15:
    print(
        "WARNING: less than 15 GiB is free. The full cache and model set "
        "may require additional Drive capacity."
    )

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    [
        "git",
        "clone",
        "--quiet",
        "--depth",
        "1",
        "--branch",
        REPO_REF,
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

src_dir = str(REPO_DIR / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
importlib.invalidate_caches()
os.chdir(REPO_DIR)

import torch
import retinal_color_transfer

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required. In Colab select Runtime -> Change runtime type -> GPU."
    )

print("Repository:", Path.cwd())
print("Package:", retinal_color_transfer.__file__)
print("CUDA:", torch.cuda.get_device_name(0))
print("Persistent run root:", RUN_ROOT)


## 4. Validate the raw dataset

The repository owns the split definitions and labels. The user supplies only
the source images. Strict validation requires every manifest path to resolve
under `DATA_ROOT` and verifies that the three splits are patient-disjoint.

The repository does not contain a checksum manifest for the private raw
images, so this step cannot prove that user-supplied image bytes are identical
to the original study data. Exact reproduction requires the same source dataset.


In [ ]:
from retinal_color_transfer.data import (
    load_manifest,
    validate_split_collection,
)

MANIFEST_ROOT = REPO_DIR / "data" / "manifests"

manifests = [
    load_manifest(
        MANIFEST_ROOT / f"{role}.csv",
        role=role,
        data_root=DATA_ROOT,
        strict_paths=True,
    )
    for role in ("train", "validation", "test")
]
validate_split_collection(manifests)
split_counts = {manifest.role: len(manifest.frame) for manifest in manifests}
expected_split_counts = {"train": 6902, "validation": 1493, "test": 1462}
if split_counts != expected_split_counts:
    raise RuntimeError(
        f"Unexpected manifest counts: {split_counts}; "
        f"expected {expected_split_counts}"
    )
print("Validated raw images:", sum(split_counts.values()))
print("Split counts:", split_counts)


## 5. Prepare caches, normalization, and experiment configs

This invokes the repository's first numbered workflow. It creates the
prepared RGB cache first and derives all remaining full-color, single-channel,
grayscale, and Custom3 representations, validates train-only normalization,
and writes all 24 experiment configs. Existing valid cache entries are reused,
so this cell is safe to rerun after a disconnect.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/01_prepare_training.py",
        "--data-root",
        str(DATA_ROOT),
        "--cache-root",
        str(CACHE_ROOT),
        "--manifest-dir",
        str(MANIFEST_ROOT),
        "--normalization-dir",
        "data/normalization",
        "--config-output-dir",
        str(GENERATED_CONFIG_ROOT),
        "--model-root",
        str(MODEL_ROOT),
        "--batch-size",
        str(BATCH_SIZE),
        "--epochs",
        str(EPOCHS),
        "--num-workers",
        str(NUM_WORKERS),
        "--prefetch-factor",
        str(PREFETCH_FACTOR),
        "--mixed-precision",
        MIXED_PRECISION,
        "--memory-format",
        MEMORY_FORMAT,
        "--device",
        "cuda",
    ],
    check=True,
)
print("All representation artifacts and 24 experiment configs are ready.")


## 6. Verify all 24 experiment configs

The shared training template remains authoritative. The generated configs differ
only in representation, seed, and the user-selected data/cache/output roots.
The cell checks every active path and validates the locked experiment schema
before training starts.


In [ ]:
import yaml

from retinal_color_transfer.artifacts import (
    representation_family,
    resolve_model_dir,
    resolve_representation_config_path,
)
from retinal_color_transfer.config import (
    RepresentationConfig,
    validate_experiment_config,
)


CONFIG_PATHS = {}
for representation, seed in MODELS:
    model_name = f"{representation}_seed{seed}"
    config_path = GENERATED_CONFIG_ROOT / f"{model_name}.yaml"
    if not config_path.is_file():
        raise FileNotFoundError(f"Missing generated config: {config_path}")
    with config_path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    validate_experiment_config(config)

    family = representation_family(representation)
    expected_paths = {
        "experiment.output_dir": str(resolve_model_dir(MODEL_ROOT, model_name)),
        "data.data_root": str(DATA_ROOT),
        "cache.base_rgb_root": str(CACHE_ROOT / "rgb" / "rgb"),
        "cache.representation_root": str(CACHE_ROOT / family / representation),
        "representation_config": str(
            resolve_representation_config_path("configs", representation)
        ),
        "normalization.statistics_path": str(
            Path("data")
            / "normalization"
            / family
            / f"{representation}_train_stats.json"
        ),
    }
    observed_paths = {
        "experiment.output_dir": config["experiment"]["output_dir"],
        "data.data_root": config["data"]["data_root"],
        "cache.base_rgb_root": config["cache"]["base_rgb_root"],
        "cache.representation_root": config["cache"]["representation_root"],
        "representation_config": config["representation_config"],
        "normalization.statistics_path": config["normalization"]["statistics_path"],
    }
    if observed_paths != expected_paths:
        raise RuntimeError(
            f"Path drift in {config_path}: "
            f"expected {expected_paths}, got {observed_paths}"
        )
    if config["runtime"]["seed"] != seed:
        raise RuntimeError(f"Seed drift in {config_path}")
    if config["optimization"]["epochs"] != EPOCHS:
        raise RuntimeError(f"Epoch drift in {config_path}")
    if config["checkpoint_selection"] != {
        "metric": "validation_mae",
        "mode": "min",
        "strict_improvement": True,
    }:
        raise RuntimeError(f"Checkpoint-selection drift in {config_path}")
    RepresentationConfig.from_file(
        config["representation_config"],
        require_approved=True,
    )
    CONFIG_PATHS[model_name] = config_path

print(f"Verified {len(CONFIG_PATHS)} generated experiment configs.")


## 7. Train all models

Models are processed sequentially. A completed training contains
`best_checkpoint.pt` and `training_history.csv` and is skipped on later runs.
An unfinished model resumes when `latest_checkpoint.pt` exists. The latest
checkpoint is transient and the training engine removes it after epoch 80.

If a model directory is incomplete but has no resumable checkpoint, the
notebook stops instead of deleting it automatically.


In [ ]:
import csv

from retinal_color_transfer.training.engine import run_training

TRAINING_FILES = {
    "best_checkpoint.pt",
    "training_history.csv",
}


def model_dir_for(model_name: str) -> Path:
    return resolve_model_dir(MODEL_ROOT, model_name)


def files_in(model_dir: Path) -> set[str]:
    if not model_dir.is_dir():
        return set()
    return {path.name for path in model_dir.iterdir() if path.is_file()}


def training_is_complete(model_dir: Path) -> bool:
    history_path = model_dir / "training_history.csv"
    if not (model_dir / "best_checkpoint.pt").is_file() or not history_path.is_file():
        return False
    with history_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    return (
        len(rows) == EPOCHS
        and max(int(row["epoch"]) for row in rows) == EPOCHS
    )


for index, (representation, seed) in enumerate(MODELS, start=1):
    model_name = f"{representation}_seed{seed}"
    model_dir = model_dir_for(model_name)
    print(f"\n[{index}/{len(MODELS)}] {model_name}", flush=True)

    if training_is_complete(model_dir):
        print("Training already complete; skipping.")
        continue

    current_files = files_in(model_dir)
    has_resume_checkpoint = (model_dir / "latest_checkpoint.pt").is_file()
    has_completed_training = training_is_complete(model_dir)
    if current_files and not has_resume_checkpoint and not has_completed_training:
        raise RuntimeError(
            f"{model_dir} is incomplete but has no latest_checkpoint.pt. "
            "Inspect or remove only this incomplete directory before rerunning."
        )

    if not has_completed_training:
        training_summary = run_training(
            CONFIG_PATHS[model_name],
            resume_if_available=True,
        )
        print(training_summary)

    if not training_is_complete(model_dir):
        raise RuntimeError(
            f"Training did not complete cleanly in {model_dir}"
        )
    unexpected = files_in(model_dir) - (TRAINING_FILES | {"predictions.csv"})
    if unexpected:
        raise RuntimeError(
            f"Unexpected training files in {model_dir}: {sorted(unexpected)}"
        )
    print("Training complete:", model_dir)

print("\nAll requested training jobs are complete.")


## 8. Create missing predictions, analyze, and audit

The second numbered workflow reuses every valid `predictions.csv`, creates
only missing predictions, and then runs the canonical metrics, bootstrap, and
Color4-vs-RGB4 diversity analyses. The final audit verifies all 24 artifacts.


In [ ]:
import json
import math

import pandas as pd

from retinal_color_transfer.evaluation import (
    validate_checkpoint_compatibility,
)

ANALYSIS_ROOT = RUN_ROOT / "analysis" / "final_results"
subprocess.run(
    [
        sys.executable,
        "scripts/02_evaluate_and_analyze.py",
        "--model-root",
        str(MODEL_ROOT),
        "--manifest-dir",
        str(MANIFEST_ROOT),
        "--data-root",
        str(DATA_ROOT),
        "--device",
        "cuda",
        "--batch-size",
        str(BATCH_SIZE),
        "--output-dir",
        str(ANALYSIS_ROOT),
    ],
    check=True,
)

FINAL_FILES = TRAINING_FILES | {"predictions.csv"}

summary_rows = []
for representation, seed in MODELS:
    model_name = f"{representation}_seed{seed}"
    model_dir = model_dir_for(model_name)
    current_files = files_in(model_dir)
    if current_files != FINAL_FILES:
        raise RuntimeError(
            f"Incomplete artifact {model_dir}: {sorted(current_files)}"
        )

    history = pd.read_csv(model_dir / "training_history.csv")
    predictions = pd.read_csv(model_dir / "predictions.csv")
    checkpoint = torch.load(
        model_dir / "best_checkpoint.pt",
        map_location="cpu",
        weights_only=False,
    )

    if len(history) != EPOCHS or int(history["epoch"].max()) != EPOCHS:
        raise RuntimeError(f"Incomplete {model_name} training history")
    split_counts = predictions["split"].value_counts().to_dict()
    if split_counts != {"validation": 1493, "test": 1462}:
        raise RuntimeError(
            f"Unexpected prediction counts for {model_name}: {split_counts}"
        )
    if predictions["image_id"].duplicated().any():
        raise RuntimeError(f"Duplicate prediction image IDs in {model_name}")

    best_row = history.loc[history["mae"].idxmin()]
    if int(checkpoint["epoch"]) != int(best_row["epoch"]):
        raise RuntimeError(f"Best-epoch mismatch in {model_name}")
    if not math.isclose(
        float(checkpoint["model_selection"]["value"]),
        float(best_row["mae"]),
        rel_tol=0.0,
        abs_tol=1e-6,
    ):
        raise RuntimeError(f"Best-MAE mismatch in {model_name}")
    if checkpoint["representation"]["name"] != representation:
        raise RuntimeError(f"Representation mismatch in {model_name}")
    if int(checkpoint["runtime"]["seed"]) != seed:
        raise RuntimeError(f"Seed mismatch in {model_name}")

    config = checkpoint["resolved_config"]
    rep_config = RepresentationConfig.from_file(
        config["representation_config"],
        require_approved=True,
    )
    normalization_path = Path(config["normalization"]["statistics_path"])
    with normalization_path.open("r", encoding="utf-8") as handle:
        normalization = json.load(handle)
    validate_checkpoint_compatibility(
        checkpoint,
        config,
        normalization,
        rep_config,
    )

    validation_mae = predictions.loc[
        predictions["split"] == "validation",
        "absolute_error",
    ].mean()
    test_mae = predictions.loc[
        predictions["split"] == "test",
        "absolute_error",
    ].mean()
    summary_rows.append(
        {
            "model": model_name,
            "validation_mae": round(float(validation_mae), 4),
            "test_mae": round(float(test_mae), 4),
            "best_epoch": int(best_row["epoch"]),
        }
    )

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
print(f"\nAudit complete: {len(summary)} canonical model artifacts are valid.")


## 9. Train Ensemble² — 12-Channel Early Fusion Model

This section trains the single ResNet-50 that receives all four color spaces
concatenated as a 12-channel input (RGB + Lab + HSV + YCrCb).  
It reuses the caches already built in Section 5 and writes the model to
`MODEL_ROOT/fusion/rgb_lab_hsv_ycrcb_seed42/`.

The cell is idempotent: if training is already complete it is skipped.
An interrupted run resumes automatically from `latest_checkpoint.pt`.


In [ ]:
FUSION_MODEL_DIR = MODEL_ROOT / "fusion" / "rgb_lab_hsv_ycrcb_seed42"
FUSION_MODEL_DIR.mkdir(parents=True, exist_ok=True)

fusion_best    = FUSION_MODEL_DIR / "best_checkpoint.pt"
fusion_history = FUSION_MODEL_DIR / "training_history.csv"

fusion_complete = (
    fusion_best.is_file()
    and fusion_history.is_file()
    and len(pd.read_csv(fusion_history)) == EPOCHS
)

if fusion_complete:
    print("Ensemble² training already complete; skipping.")
else:
    subprocess.run(
        [
            sys.executable, "scripts/train_fusion12ch.py",
            "--cache-root",          str(CACHE_ROOT),
            "--model-dir",           str(FUSION_MODEL_DIR),
            "--device",              "cuda",
            "--epochs",              str(EPOCHS),
            "--batch-size",          str(BATCH_SIZE),
            "--num-workers",         str(NUM_WORKERS),
            "--seed",                "42",
            "--mixed-precision",     MIXED_PRECISION,
            "--allow-weight-download",
        ],
        check=True,
    )

# Summary
fusion_preds = pd.read_csv(FUSION_MODEL_DIR / "predictions.csv")
for split in ["validation", "test"]:
    mae = fusion_preds[fusion_preds["split"] == split]["absolute_error"].mean()
    print(f"Ensemble² {split} MAE: {mae:.4f}")
